#sujet 1.1 : optimisartion du transport

In [16]:

import pulp

# Données du problème
capacites = [100, 150]           # Usines 1 et 2
demandes = [80, 120, 50]         # Entrepôts 1,2,3
cout = [[4, 2, 6],               # coût unitaire de l'usine 1 vers chaque entrepôt
        [3, 5, 4]]               # usine 2

# Création du problème
prob = pulp.LpProblem("Transport_Optimisation", pulp.LpMinimize)

# Variables de décision : x[i][j] = quantité de l'usine i à l'entrepôt j
x = [[pulp.LpVariable(f"x_{i+1}_{j+1}", lowBound=0, cat='Continuous')
      for j in range(3)] for i in range(2)]

# Fonction objectif
prob += pulp.lpSum(cout[i][j] * x[i][j] for i in range(2) for j in range(3))

# Contraintes de capacité des usines
for i in range(2):
    prob += pulp.lpSum(x[i][j] for j in range(3)) <= capacites[i]

# Contraintes de satisfaction des demandes
for j in range(3):
    prob += pulp.lpSum(x[i][j] for i in range(2)) == demandes[j]

# Résolution
prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Affichage des résultats
print("Statut :", pulp.LpStatus[prob.status])
print("Solution optimale :")
for i in range(2):
    for j in range(3):
        val = x[i][j].varValue
        if val > 0:
            print(f"  x_{i+1}_{j+1} = {val}")
print(f"Coût total = {pulp.value(prob.objective)}")

Statut : Optimal
Solution optimale :
  x_1_2 = 100.0
  x_2_1 = 80.0
  x_2_2 = 20.0
  x_2_3 = 50.0
Coût total = 740.0


# sujet 2.1 : TSP

In [14]:

from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# Matrice des distances (5 villes A,B,C,D,E)
dist_matrix = [
    [0, 10, 15, 20, 25],
    [10, 0, 35, 25, 30],
    [15, 35, 0, 30, 20],
    [20, 25, 30, 0, 15],
    [25, 30, 20, 15, 0]
]

def solve_tsp():
    # Création du modèle
    manager = pywrapcp.RoutingIndexManager(len(dist_matrix), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    # Fonction de coût (distance)
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Paramètres de recherche
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        # Affichage du trajet
        index = routing.Start(0)
        route = []
        while not routing.IsEnd(index):
            route.append(chr(65 + manager.IndexToNode(index)))  # A,B,C,...
            index = solution.Value(routing.NextVar(index))
        route.append(chr(65 + manager.IndexToNode(index)))
        print("Trajet optimal :", " → ".join(route))
        print("Distance totale :", solution.ObjectiveValue(), "km")
    else:
        print("Aucune solution trouvée.")

if __name__ == "__main__":
    solve_tsp()


Trajet optimal : A → B → D → E → C → A
Distance totale : 85 km


# sujet 6.3 : Routage des camions d'eau

In [15]:

from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# Données réelles approximatives
depot = 0
demandes = [0, 300, 400, 200, 500, 300, 250]   # en litres (indice 0 = dépôt)
capacite_vehicule = 1000  # litres
nb_vehicules = 2

# Matrice des distances (km) symétrique
dist_matrix = [
    [0, 5, 7, 10, 12, 8, 9],
    [5, 0, 4, 6, 8, 5, 7],
    [7, 4, 0, 5, 9, 6, 4],
    [10, 6, 5, 0, 7, 8, 6],
    [12, 8, 9, 7, 0, 10, 9],
    [8, 5, 6, 8, 10, 0, 3],
    [9, 7, 4, 6, 9, 3, 0]
]

def solve_vrp():
    manager = pywrapcp.RoutingIndexManager(len(dist_matrix), nb_vehicules, depot)
    routing = pywrapcp.RoutingModel(manager)

    # Fonction distance
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Contrainte de capacité
    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return demandes[from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # slack
        [capacite_vehicule] * nb_vehicules,
        True,
        'Capacity'
    )

    # Paramètres de recherche
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        total_distance = 0
        for v in range(nb_vehicules):
            index = routing.Start(v)
            route = []
            charge = 0
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                route.append(node)
                charge += demandes[node]
                prev = index
                index = solution.Value(routing.NextVar(index))
                total_distance += routing.GetArcCostForVehicle(prev, index, v)
            route.append(manager.IndexToNode(index))
            print(f"Camion {v+1} : trajet {route} -> charge {charge} L")
        print(f"Distance totale parcourue : {total_distance} km")
    else:
        print("Aucune solution trouvée.")

if __name__ == "__main__":
    solve_vrp()

Camion 1 : trajet [0, 5, 3, 4, 0] -> charge 1000 L
Camion 2 : trajet [0, 1, 2, 6, 0] -> charge 950 L
Distance totale parcourue : 57 km
